# 3D Vision — Rays and Volume Rendering

This notebook executes the local NumPy ray and volume-rendering artifact. It does not download or train a NeRF.

In [ ]:
from pathlib import Path
import importlib.util

lesson_rel = Path('phases/04-computer-vision/13-3d-vision-nerf')
candidates = [
    Path.cwd() / lesson_rel / 'code' / 'main.py',
    Path.cwd() / 'code' / 'main.py',
    Path.cwd().parent / 'code' / 'main.py',
    Path.cwd().parent.parent / 'code' / 'main.py',
]
main_path = next((p.resolve() for p in candidates if p.is_file()), None)
if main_path is None or main_path.parent.parent.name != '13-3d-vision-nerf':
    raise FileNotFoundError(f'lesson entrypoint not found; checked {candidates}')
spec = importlib.util.spec_from_file_location('cv04_l13_notebook', main_path)
lesson = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(lesson)
print(f'loaded {main_path}')

In [ ]:
import numpy as np
origins = np.array([[0.0, 0.0, 0.0], [0.1, 0.0, 0.0]])
directions = np.array([[0.0, 0.0, 1.0], [0.0, 0.1, 1.0]])
points, depths = lesson.sample_ray_points(origins, directions, 1.0, 3.0, 5)
encoded = lesson.positional_encoding(points.reshape(-1, 3), levels=3)
print('ray points:', points.shape, 'depths:', depths.shape, 'encoding:', encoded.shape)

In [ ]:
sigma, _ = lesson.density_fixture(depths)
rgb = np.full((*sigma.shape, 3), 0.4)
rendered, depth, weights = lesson.volume_render(sigma, rgb, depths)
assert np.isfinite(rendered).all() and np.isfinite(depth).all()
assert np.all(weights >= 0.0)
print('rendered:', rendered.shape, 'depth:', depth.round(4), 'weight sums:', weights.sum(axis=-1).round(4))